# Data Analysis of the filtered Bookings dataset

---

In [1]:
import os

import duckdb
import matplotlib.pyplot as plt
from dython.nominal import associations

In [2]:
# File path of filtered parquet
DATA_DIR = r"D:\Universität\Master\Semester2\RealWorld_ML_Problems\Data"
filtered_bookings_file = os.path.join(DATA_DIR, "bookings_encoded_data.parquet")
HEATMAP_FILE = os.path.join(DATA_DIR, "correlation_heatmap.png")

con = duckdb.connect()

In [3]:
df = con.execute(f"SELECT * FROM '{filtered_bookings_file}'").fetchdf()
print(f"Shape: {df.shape}")
print("Summary:")
print(df.describe(include='all'))
print("Column types:")
print(df.dtypes)

Shape: (1446575, 221)
Summary:
       workstep_number_mes  sequence_number_1679091c  \
count         1.446575e+06              1.446575e+06   
mean          1.675049e-16             -2.505067e-19   
std           1.000000e+00              1.000000e+00   
min          -1.609767e+00             -4.477472e-03   
25%          -1.085639e+00             -4.477472e-03   
50%          -3.738202e-02             -4.477472e-03   
75%           1.010875e+00             -4.477472e-03   
max           1.535003e+00              2.233403e+02   

       sequence_number_45c48cce  sequence_number_8f14e45f  \
count              1.446575e+06              1.446575e+06   
mean               1.105177e-19             -9.897472e-19   
std                1.000000e+00              1.000000e+00   
min               -1.175831e-03             -2.997803e-03   
25%               -1.175831e-03             -2.997803e-03   
50%               -1.175831e-03             -2.997803e-03   
75%               -1.175831e-03      

In [4]:
def plot_and_save_heatmap(df, output_path):
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    plt.figure(figsize=(20, 18))
    associations(df[numeric_cols], nominal_columns=[],
                 mark_columns=True, figsize=(20, 18),
                 cmap='coolwarm', annot=True, fmt='.2f',
                 plot=True, title="Correlation Heatmap")
    plt.title("Correlation Heatmap")
    plt.tight_layout()
    plt.savefig(output_path, dpi=300)
    plt.close()

print("Generating correlation heatmap...")
plot_and_save_heatmap(filtered_bookings_file, HEATMAP_FILE)
print(f"Heatmap saved to {HEATMAP_FILE}")

   book_state    count
0           0  1438810
1           1     5637
2           2     2128


### Time Difference Analysis

In [4]:
con.execute(f"""
CREATE OR REPLACE TEMP TABLE temp_bookings AS
SELECT *, EPOCH(book_stamp) - EPOCH(created_at) AS time_diff
FROM '{filtered_bookings_file}';
""")

summary = con.execute("""
SELECT
    COUNT(*) AS total,
    COUNT(*) FILTER (WHERE time_diff = 0) AS zero_diff_count,
    MIN(time_diff) AS min_diff,
    MAX(time_diff) AS max_diff,
    AVG(time_diff) AS mean_diff,
    STDDEV_SAMP(time_diff) AS std_diff
FROM temp_bookings;
""").fetchdf()
print("\n📊 Time Difference Summary:\n", summary)


📊 Time Difference Summary:
      total  zero_diff_count  min_diff   max_diff  mean_diff    std_diff
0  1446575                0     0.001  58070.201   0.483495  118.125466


### NULL percentages per column

In [5]:
columns = con.execute("DESCRIBE temp_bookings").fetchdf()['column_name'].tolist()
total_rows = con.execute("SELECT COUNT(*) FROM temp_bookings").fetchone()[0]
null_results = []
for col in columns:
    null_count = con.execute(f"SELECT COUNT(*) - COUNT({col}) FROM temp_bookings").fetchone()[0]
    null_percentage = round(100.0 * null_count / total_rows, 2)
    null_results.append((col, null_percentage))
null_results.sort(key=lambda x: x[1], reverse=True)

print("\n=== Missing Values (% by column) ===")
for col, perc in null_results:
    print(f"{col}: {perc:.2f}%")


=== Missing Values (% by column) ===
booking_id: 0.00%
book_state: 0.00%
workstep_number_mes: 0.00%
created_at: 0.00%
book_stamp: 0.00%
serial_number_id: 0.00%
station_id: 0.00%
part_group: 0.00%
line_id: 0.00%
time_diff: 0.00%


### Cardinality (# of unique values per column)

In [6]:
print("\n=== Cardinality (Number of Unique Values) ===")
for col in columns:
    unique_count = con.execute(f"SELECT COUNT(DISTINCT {col}) FROM temp_bookings").fetchone()[0]
    print(f"{col}: {unique_count}")


=== Cardinality (Number of Unique Values) ===
booking_id: 1446345
book_state: 3
workstep_number_mes: 7
created_at: 887376
book_stamp: 1446169
serial_number_id: 309483
station_id: 11
part_group: 10
line_id: 2
time_diff: 4019


### Top 10 frequent values for categorical/object columns

In [7]:
categorical_cols = ['part_group', 'line_id', 'serial_number_id', 'book_state', 'station_id']
print("\n=== Top 10 Frequent Values Per Categorical Column ===")
for col in categorical_cols:
    print(f"\n--- {col} ---")
    freq = con.execute(f"""
        SELECT {col}, COUNT(*) AS freq
        FROM temp_bookings
        GROUP BY {col}
        ORDER BY freq DESC
        LIMIT 10;
    """).fetchdf()
    print(freq)

# === Schema ===
schema = con.execute("DESCRIBE temp_bookings").fetchdf()
print("\n=== Dataset Schema ===\n", schema)


=== Top 10 Frequent Values Per Categorical Column ===

--- part_group ---
  part_group    freq
0   7a78616d  511939
1   faa60612  430133
2   d5fd2a7a  171702
3   39d0d72e  147408
4   8db45195   74057
5   f14cc036   41710
6   f8d9094d   38103
7   6db88e1e   25065
8   83e223f1    6452
9   42939de2       6

--- line_id ---
    line_id     freq
0  b1d852cd  1340995
1  ad63c958   105580

--- serial_number_id ---
  serial_number_id  freq
0         d7273b00    14
1         43e70227    14
2         0a955706    14
3         e8102b55    13
4         b4ca18cf    11
5         1b56f7fb    11
6         ad0d3da9    11
7         c6037fa2    11
8         8dc19bcd    11
9         1e0814c0    11

--- book_state ---
   book_state     freq
0           0  1438810
1           1     5637
2           2     2128

--- station_id ---
  station_id    freq
0   a78230b0  210955
1   904d6ebc  206003
2   450f2075  205803
3   2435c59b  205091
4   bbee591d  205062
5   6ecd370a  204864
6   c61cc047  103217
7   38b291ac 

In [8]:
con.close()